# 14. Naive Bayes
**Use case:** Financial Services — Customer complaint routing to the right product team

Banks receive thousands of written complaints each month through web forms, email and regulator portals such as the CFPB complaint database. Each must reach the right team - credit cards, mortgage servicing, fraud or fees - quickly, because regulators expect timely responses and misrouted fraud cases can cost customers money. Manual triage is slow and inconsistent, but years of labelled complaint narratives already exist. Success means most complaints are routed automatically and accurately, fraud cases are rarely missed, and uncertain cases go to a human triage queue.

> 💡 A bank call-centre veteran hears 'escrow' and thinks mortgage, hears 'unauthorized' and thinks fraud, and weighs all the clue words together to pick a team. They treat each clue separately rather than reasoning about how clues interact - that is the 'naive' part, and it still works surprisingly well.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Count words and apply Bayes by hand
Build a word-count Naive Bayes classifier in pure Python on 8 complaints and route a new one.

| 5W + H | |
|---|---|
| **Who** | A complaints operations analyst learning how text classifiers decide. |
| **What** | Eight labelled complaints across Credit Card, Mortgage, Fraud and Fees, and one new complaint to route. |
| **When** | Before using scikit-learn, to see the probability arithmetic. |
| **Where** | A plain Python script using only the standard library. |
| **Why** | Understanding priors, word likelihoods and smoothing explains why the model makes each routing call. |
| **How** | Count words per class, compute smoothed log-probabilities, add them up and pick the highest. |

**🧠 ML concepts:** Prior probability, Likelihood P(word | class), Laplace smoothing, Working in log space, Turning scores into probabilities  
**🐍 Python concepts:** List of tuples, Set and dict comprehensions, collections.Counter, Functions with default arguments, sorted with a key lambda, max with key=dict.get

In [ ]:
import math
from collections import Counter

# 8 short training complaints, each already labelled by a human agent
# 🐍 PY: a list of (text, label) tuples - a simple way to keep data and answers together
train = [
    ("charged twice on my credit card statement", "Credit Card"),
    ("credit card interest rate went up", "Credit Card"),
    ("mortgage escrow payment is wrong", "Mortgage"),
    ("loan servicer lost my mortgage payment", "Mortgage"),
    ("someone used my card without permission fraud", "Fraud"),
    ("unauthorized transfer from my account fraud", "Fraud"),
    ("overdraft fee charged again", "Fees"),
    ("monthly maintenance fee on my account", "Fees"),
]

# 🐍 PY: a set comprehension collects each label once
labels = sorted({label for _, label in train})
# 🐍 PY: Counter counts word occurrences; dict comprehension makes one Counter per class
word_counts = {c: Counter() for c in labels}
for text, label in train:
    word_counts[label].update(text.split())
vocab = {w for c in labels for w in word_counts[c]}
print("Vocabulary size:", len(vocab))
print("Top words in Fraud:", word_counts["Fraud"].most_common(3))

# 🧠 ML: prior = how common each class is before reading the text (here 2/8 each)
priors = {c: sum(1 for _, l in train if l == c) / len(train) for c in labels}

def class_score(text, c, alpha=1.0):
    # 🧠 ML: Bayes rule in log form: log P(class) + sum of log P(word | class)
    score = math.log(priors[c])
    total = sum(word_counts[c].values())
    for w in text.split():
        # 🧠 ML: Laplace smoothing (+alpha) stops an unseen word from making the probability zero
        p = (word_counts[c][w] + alpha) / (total + alpha * len(vocab))
        score += math.log(p)
    return score

new_complaint = "unauthorized charge on my credit card"
print(f"\nNew complaint: '{new_complaint}'")
# 🐍 PY: dict comprehension scores every class; max(..., key=...) picks the best one
scores = {c: class_score(new_complaint, c) for c in labels}
for c, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"  {c:<12} log-score {s:7.2f}")

# 🧠 ML: convert log-scores to probabilities that sum to 1 (softmax of the logs)
m = max(scores.values())
exp = {c: math.exp(s - m) for c, s in scores.items()}
probs = {c: v / sum(exp.values()) for c, v in exp.items()}
best = max(probs, key=probs.get)
print(f"Route to: {best} team (probability {probs[best]:.0%})")

**Try this next:**
- Add 'unauthorized' to one Credit Card example and see whether the routing flips.
- Set alpha to 0.01 and watch how much more extreme the probabilities become.
- Add a third Fraud example and see how the prior and likelihoods change.

## BUILD: CountVectorizer plus MultinomialNB on 3,000 complaints
Train a bag-of-words Multinomial Naive Bayes on 3,000 synthetic complaints and measure routing accuracy.

| 5W + H | |
|---|---|
| **Who** | A data analyst supporting the complaints management team. |
| **What** | 3,000 synthetic CFPB-style narratives with mixed topics and some mislabelled cases. |
| **When** | When setting up a first automated routing model to replace manual triage. |
| **Where** | A Python job reading exported complaint narratives. |
| **Why** | Naive Bayes trains in seconds, needs little tuning and is easy to explain to compliance. |
| **How** | CountVectorizer turns text into word counts, MultinomialNB learns word probabilities per team, and confident predictions are auto-routed. |

**🧠 ML concepts:** Bag-of-words features, Multinomial Naive Bayes, Stratified split for multi-class data, Precision, recall and F1 per class, Confidence-based routing  
**🐍 Python concepts:** Dict of lists, List comprehension with a filter, rng.choice and in-place shuffle, str.join to build text, zip over predictions, Boolean masks for subsets

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

rng = np.random.default_rng(42)

# --- 1. Generate 3,000 synthetic consumer complaints (CFPB-style narratives) ---
# 🐍 PY: a dict of product -> list of phrases typical for that product
phrases = {
    "Credit Card": ["charged twice on my card", "credit limit was lowered", "interest rate increased",
                    "card statement balance wrong", "rewards points not credited", "minimum payment"],
    "Mortgage": ["escrow account shortage", "loan modification denied", "mortgage payment misapplied",
                 "servicer foreclosure notice", "refinance closing delayed", "property tax in escrow"],
    "Fraud": ["unauthorized transaction", "identity theft account opened", "someone stole my card",
              "scam wire transfer", "account hacked", "did not authorize this charge"],
    "Fees": ["overdraft fee charged", "monthly maintenance fee", "late fee unfair",
             "atm fee refunded never", "wire fee too high", "fee charged without notice"],
}
# 🧠 ML: shared filler words appear in every class - real complaints are full of them
filler = ["i called the bank", "please help", "this is the third time", "my account",
          "they will not respond", "very frustrated", "customer service", "on my statement"]
# 🧠 ML: real complaints mix topics (a fee on a card, fraud on a mortgage) - this keeps the task honest
products = list(phrases)

rows = []
# 🐍 PY: zip pairs each class with its share of the complaint volume
for product, share in zip(products, [0.35, 0.25, 0.20, 0.20]):
    for _ in range(int(3000 * share)):
        words = list(rng.choice(phrases[product], 2)) + list(rng.choice(filler, 3))
        if rng.random() < 0.4:
            other = rng.choice([p for p in products if p != product])
            words.append(rng.choice(phrases[other]))
        # 🧠 ML: about 8% of consumers pick the wrong product on the form - label noise
        label = rng.choice(products) if rng.random() < 0.08 else product
        # 🐍 PY: shuffle in place, then " ".join glues the phrases into one narrative string
        rng.shuffle(words)
        rows.append({"complaint_text": " ".join(words), "product": label})
df = pd.DataFrame(rows)
print(df.sample(3, random_state=1).to_string(index=False))
print("\nClass counts:\n" + df["product"].value_counts().to_string())

# --- 2. Split ---
X_train, X_test, y_train, y_test = train_test_split(
    df["complaint_text"], df["product"], test_size=0.25, stratify=df["product"], random_state=42)

# --- 3. Turn text into word counts ---
# 🧠 ML: bag-of-words - each complaint becomes a row of counts, one column per vocabulary word
vectorizer = CountVectorizer(stop_words="english")
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)
print(f"\nVocabulary: {len(vectorizer.vocabulary_)} words; matrix shape {Xtr.shape}")

# --- 4. Train Multinomial Naive Bayes ---
# 🧠 ML: alpha=1.0 is Laplace smoothing, same idea as the hand-made easy level
nb = MultinomialNB(alpha=1.0)
nb.fit(Xtr, y_train)
pred = nb.predict(Xte)
print(f"\nTest accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, digits=3))

# --- 5. Route new complaints ---
new = ["someone opened a card in my name identity theft",
       "they charged an overdraft fee twice this month",
       "my escrow went up and the mortgage payment doubled"]
# 🐍 PY: predict_proba returns one row per complaint; argmax picks the most likely class index
proba = nb.predict_proba(vectorizer.transform(new))
for text, p in zip(new, proba):
    print(f"{nb.classes_[p.argmax()]:<12} {p.max():.0%}  <- '{text}'")

# 🧠 ML: confidence-based routing - low-confidence cases go to a human triage queue
test_conf = nb.predict_proba(Xte).max(axis=1)
auto = test_conf >= 0.90
print(f"\nAuto-route {auto.mean():.0%} of complaints at >=90% confidence; "
      f"accuracy on those {accuracy_score(y_test[auto], pred[auto]):.1%}")

**Try this next:**
- Remove stop_words='english' and compare the vocabulary size and accuracy.
- Set the label-noise rate from 0.08 to 0 and see accuracy rise.
- Change alpha to 0.01 and 10 and compare the per-team F1 scores.

## PRODUCTION: Tuned TF-IDF Naive Bayes router with fraud escalation
Tune a TF-IDF plus Naive Bayes pipeline with grid search, compare MultinomialNB and ComplementNB on imbalanced data, explain it and set a routing threshold.

| 5W + H | |
|---|---|
| **Who** | The complaints analytics team, with compliance and fraud operations as stakeholders. |
| **What** | A saved routing pipeline with tuned n-grams and smoothing, per-team words, a confusion matrix and a confidence plus fraud-escalation rule. |
| **When** | Retrained monthly on newly labelled complaints; thresholds reviewed with compliance. |
| **Where** | A model API called by the case-management system when each complaint arrives. |
| **Why** | Misrouted fraud complaints harm customers and regulators expect timely, fair complaint handling. |
| **How** | Pipeline with TfidfVectorizer and NB, 5-fold stratified GridSearchCV on macro-F1, dummy baseline, model choice by CV score, confusion matrix, top words and a threshold table. |

**🧠 ML concepts:** TF-IDF with n-grams, Grid search with stratified cross-validation, Macro-F1 for imbalanced classes, MultinomialNB vs ComplementNB, Confusion matrix analysis, Explaining NB with per-class word probabilities  
**🐍 Python concepts:** Module-level constants in CAPS, Passing a class as a function argument, isinstance checks, Dict comprehension over enumerate, max with a key lambda, joblib and tempfile for saving

In [ ]:
"""Complaint routing: TF-IDF + Naive Bayes pipeline, grid search over alpha/ngrams,
MultinomialNB vs ComplementNB on imbalanced classes, confusion matrix, top words, routing threshold."""
import os
import tempfile
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
PHRASES = {
    "Credit Card": ["charged twice on my card", "credit limit was lowered", "interest rate increased",
                    "card statement balance wrong", "rewards points not credited", "minimum payment due",
                    "balance transfer promo", "card payment posted late"],
    "Mortgage": ["escrow account shortage", "loan modification denied", "mortgage payment misapplied",
                 "servicer foreclosure notice", "refinance closing delayed", "property tax in escrow",
                 "pmi not removed", "payment posted late to loan"],
    "Fraud": ["unauthorized transaction", "identity theft account opened", "someone stole my card",
              "scam wire transfer", "account hacked", "did not authorize this charge",
              "phishing call took my pin", "charged twice by unknown merchant"],
    "Fees": ["overdraft fee charged", "monthly maintenance fee", "late fee unfair",
             "atm fee never refunded", "wire fee too high", "fee charged without notice",
             "paper statement fee", "returned item fee"],
}
FILLER = ["i called the bank", "please help", "this is the third time", "my account", "on my statement",
          "they will not respond", "very frustrated", "customer service", "i want a refund", "since last month"]
# 🧠 ML: imbalanced like real complaint volumes - fraud is the rarest yet most urgent class
SHARES = {"Credit Card": 0.42, "Mortgage": 0.28, "Fees": 0.20, "Fraud": 0.10}


def make_complaints(n=4000):
    products, rows = list(PHRASES), []
    for product, share in SHARES.items():
        for _ in range(int(n * share)):
            words = list(rng.choice(PHRASES[product], 2)) + list(rng.choice(FILLER, 3))
            if rng.random() < 0.4:
                other = rng.choice([p for p in products if p != product])
                words.append(rng.choice(PHRASES[other]))
            label = rng.choice(products) if rng.random() < 0.06 else product
            rng.shuffle(words)
            rows.append((" ".join(words), str(label)))
    return pd.DataFrame(rows, columns=["complaint_text", "product"])


def build_search(nb_class):
    # 🧠 ML: Pipeline keeps vectoriser + model together so CV never leaks test vocabulary into training
    pipe = Pipeline([("tfidf", TfidfVectorizer(stop_words="english", sublinear_tf=True)),
                     ("nb", nb_class())])
    grid = {"tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__min_df": [1, 3],
            "nb__alpha": [0.01, 0.1, 0.5, 1.0]}
    # 🧠 ML: macro-F1 treats all 4 teams equally, so the small Fraud class is not ignored
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=1)


def top_words(pipe, k=6):
    """For each class: words that class 'owns' (highest P(word|class) among classes), most probable first."""
    vocab = np.array(pipe.named_steps["tfidf"].get_feature_names_out())
    nb = pipe.named_steps["nb"]
    logp = nb.feature_log_prob_
    if isinstance(nb, ComplementNB):
        logp = -logp  # ComplementNB stores weights of the complement, so flip the sign
    # 🧠 ML: a word belongs to the class where it is most likely; then rank by how likely it is there
    owner = logp.argmax(axis=0)
    out = {}
    for i, c in enumerate(nb.classes_):
        idx = np.where(owner == i)[0]
        out[c] = list(vocab[idx[np.argsort(logp[i, idx])[::-1][:k]]])
    return out


if __name__ == "__main__":
    df = make_complaints()
    print("Class mix:", df["product"].value_counts(normalize=True).round(2).to_dict())
    X_tr, X_te, y_tr, y_te = train_test_split(df.complaint_text, df["product"], test_size=0.25,
                                              stratify=df["product"], random_state=42)

    baseline = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
    print(f"Baseline (always Credit Card) macro-F1: {f1_score(y_te, baseline.predict(X_te), average='macro'):.3f}\n")

    results = {}
    for name, cls in [("MultinomialNB", MultinomialNB), ("ComplementNB", ComplementNB)]:
        search = build_search(cls).fit(X_tr, y_tr)
        pred = search.predict(X_te)
        results[name] = search
        print(f"{name:<14} best CV macro-F1 {search.best_score_:.3f} | test macro-F1 "
              f"{f1_score(y_te, pred, average='macro'):.3f} | Fraud recall "
              f"{(pred[y_te == 'Fraud'] == 'Fraud').mean():.3f}")
        print(f"{'':<14} best params {search.best_params_}")

    # 🧠 ML: pick the winner by CV score (not test) to keep the test set honest
    best_name = max(results, key=lambda k: results[k].best_score_)
    best = results[best_name].best_estimator_
    pred = best.predict(X_te)
    print(f"\nChosen model: {best_name}")
    print(classification_report(y_te, pred, digits=3))

    labels = list(best.classes_)
    cm = pd.DataFrame(confusion_matrix(y_te, pred, labels=labels), index=labels, columns=labels)
    print("Confusion matrix (rows = true team, columns = routed team):")
    print(cm.to_string())

    print("\nMost distinctive words per team:")
    for c, words in top_words(best).items():
        print(f"  {c:<12} {', '.join(words)}")

    # 🧠 ML: business rule - auto-route only when confident; anything mentioning fraud risk is escalated
    proba = best.predict_proba(X_te)
    conf = proba.max(axis=1)
    fraud_p = proba[:, labels.index("Fraud")]
    print("\nThreshold  auto-routed  accuracy-on-auto  sent-to-human")
    for t in [0.5, 0.6, 0.7, 0.8]:
        auto = (conf >= t) & (fraud_p < 0.3)
        acc = (pred[auto] == y_te.values[auto]).mean()
        print(f"   {t:.1f}       {auto.mean():6.1%}       {acc:6.1%}         {1 - auto.mean():6.1%}")

    path = os.path.join(tempfile.mkdtemp(), "complaint_router.joblib")
    joblib.dump(best, path)
    print(f"\nSaved pipeline -> {os.path.basename(path)}")
    print("Compliance: complaints must be handled within regulatory timelines (e.g. CFPB 15/60 days);")
    print("audit misroutes monthly and keep humans on low-confidence and possible-fraud cases.")

**Try this next:**
- Add 'tfidf__ngram_range': (1, 3) to the grid and check whether trigrams help.
- Lower Fraud's share to 0.03 and compare MultinomialNB and ComplementNB Fraud recall.
- Change the fraud escalation cut-off from 0.3 to 0.1 and see how many more cases go to humans.